# HCSN：自由边界双通道的可行性与数学核验

**2026-10-30｜研究判断、独立推导与已执行的标量数值检查。**

可以把价值函数、首达风险和内生自由边界构造成可训练的神经计算模块。现有证据还不支持其优于 SNN、连续时间 RNN、状态空间模型或 Transformer。本文没有训练 HCSN，没有复现 He–Xiong 完整 equity 附录解，没有测量模型质量、硬件能耗或生产 Hepta 行为。

本文补充此前 rollover_snn_research.ipynb 和 RR_SNN_V2_Report.ipynb，重点解决本次“Call/Put 双通道都 smooth-pasting”“连续边界等于脉冲流”“解析值函数等于解析神经动力学”的三项混淆。下面所有代码单元已在当前 Python 进程依次实际执行，输出随文件保存；重跑只需 NumPy、SciPy、Matplotlib。

## 1. 原论文提供了什么

He & Xiong (2012), Journal of Finance 67(2), 391–430，在常参数、平稳债务期限结构下，用一维随机状态、债务与 equity 的耦合估值，以及 value matching / smooth pasting 求出内生违约边界。债务解见式 (8)，equity ODE 见式 (11)，边界见式 (12)，完整 equity 解析式见附录 A7。[1]

该边界在原设定中为常数；违约终止企业。将 equity 解释为回复力、将风险解释为再生力，或加入恢复与重复发放，都属于新模型定义，需要重新检验。论文中的金融风险中性概率不能直接解释为生物发放概率。

## 2. 三层解析性必须分开

1. **解析特征**：闭式 E(V,b) 或首达概率作为非线性映射；可直接自动微分。
2. **局部精确传播**：仅在其指定的漂移、噪声、边界和时间区间假设下，传播核精确。
3. **网络级闭式解**：任意输入、耦合、动态阈值及恢复状态的整个网络可精确求解。前两项不能推出第三项。

若 $x=\log V-\log b(t)$，且 b 是可微有限变差过程，则

$$dx=\left(\mu_V-\frac{\sigma^2}{2}-\frac{d\log b}{dt}\right)dt+\sigma dW.$$

仅把新 b 代入原固定系数公式，会遗漏边界移动。分段冻结可以作为明确声明的数值/建模近似，但应检验分段误差。


## 3. 一对透明的解析通道：仅为常成本简化

以下问题是独立定义的常成本最优停止简化，删除了原文期限相关的 rollover 项，**不是原文完整解析解的复现**：

$$rE=(r-\delta)VE_V+\tfrac12\sigma^2V^2E_{VV}+\delta V-K,$$
$$E(b)=E_V(b)=0,\qquad E(V)\sim V-K/r\quad(V\to\infty).$$

假设 $r,\delta,\sigma,K>0$，记

$$\nu=r-\delta-\sigma^2/2,\quad
\gamma=\frac{\nu+\sqrt{\nu^2+2r\sigma^2}}{\sigma^2}>0,\quad
b=\frac{\gamma K}{r(1+\gamma)}.$$

则

$$E(V)=V-\frac Kr+\frac b\gamma (b/V)^\gamma,\qquad
Q(V)=(b/V)^\gamma=\mathbb E_V[e^{-r\tau_b}].$$

上式 $\mathbb E_V$ 表示从 V 出发的期望。$\tau_b$ 是首次触界时刻，无穷首达时的折现支付为零。Q 是折现触界权，不能当成一个普通 put 的价格或有限期限的发放概率。

令 $z=V/b\ge1$，无量纲特征为

$$C(z)=E(bz)/b=z-\frac{\gamma+1}{\gamma}+\frac{z^{-\gamma}}{\gamma},
\quad Q(z)=z^{-\gamma}.$$

$$C(1)=C'(1)=0,\qquad Q(1)=1,\quad Q'(1)=-\gamma.$$

C 向 z≤1 零延拓时为 C1；Q 向 z≤1 常数1延拓时一般仅 C0。因此“两个通道都以零导数光滑接合”不成立。C 的一阶梯度在边界恰好消失。

两个通道均由单个 z 决定，局部特征 Jacobian 秩最多为1。双读出可能改善归纳偏置，但不会自动增加独立状态维数或信息量。


## 4. 有明确时间意义的风险算子

独立定义固定系数、固定吸收边界过程

$$dX_t=\nu dt+\sigma dW_t,\quad X_0=x>0,\quad
\tau_0=\inf\{t:X_t\le0\}.$$

对于 $T,\sigma>0$：

$$P_T(x)=\Phi\!\left(\frac{-x-\nu T}{\sigma\sqrt T}\right)
+e^{-2\nu x/\sigma^2}\Phi\!\left(\frac{-x+\nu T}{\sigma\sqrt T}\right).$$

这是区间内至少触界一次的概率。它不是区间末端低于边界的概率，也不是当前发放率。

$$f(t|x)=\partial_tP_t(x)
=\frac{x}{\sigma\sqrt{2\pi t^3}}
\exp\!\left[-\frac{(x+\nu t)^2}{2\sigma^2t}\right].$$

对零漂移：

$$\partial_xP_T(0+)=-\sqrt{2/\pi}/(\sigma\sqrt T)\ne0.$$

当 x≤0 被扩成 P=1 时又出现导数折点。若用 softplus 令 x>0 来实现全域平滑风险特征，便没有真实穿过原边界。保留光滑风险读出完全可行，但需按连续状态网络评价。

当 $\nu\le0$ 时最终触界概率为1；因此不能用无限期限的非折现 default probability 取代有限期限风险，否则整个安全域可能都输出1。

### 传播核与边界流

对 z>0，吸收转移密度为

$$K_t(z|x)=\frac{\exp[-(z-x-\nu t)^2/(2\sigma^2t)]}{\sigma\sqrt{2\pi t}}
\left(1-e^{-2xz/(\sigma^2t)}\right).$$

其质量是 $1-P_t(x)$，且满足

$$P_{t+s}(x)=P_t(x)+\int_0^\infty K_t(z|x)P_s(z)\,dz.$$

这说明跨区间推进必须保留存活状态信息，不能简单重复使用同一个标量概率。

$$j(t|x)=\frac{\sigma^2}{2}\partial_zK_t(0+|x)=f(t|x).$$

j 才是有严格定义的连续边界概率流。以下计算独立验证质量、区间组合和边界流。


In [ ]:
import math
import platform
import numpy as np
import scipy
from scipy.integrate import quad
from scipy.special import ndtr, log_ndtr
from scipy.sparse import diags
from scipy.sparse.linalg import splu
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def channels(z, gamma):
    z = np.asarray(z, dtype=float)
    if gamma <= 0 or np.any(z < 1):
        raise ValueError("Require gamma>0, z>=1.")
    logz = np.log(z)
    # Stable enough near z=1; use a Taylor branch for the cancelling linear terms.
    C = np.expm1(logz) + np.expm1(-gamma*logz)/gamma
    small = np.abs(logz) < 1e-4
    series = ((1+gamma)*logz**2/2
              +(1-gamma**2)*logz**3/6
              +(1+gamma**3)*logz**4/24
              +(1-gamma**4)*logz**5/120)
    return np.where(small, series, C), np.exp(-gamma*logz)

def risk(x, T, nu, sigma):
    x = np.asarray(x, dtype=float)
    if T <= 0 or sigma <= 0 or np.any(x < 0):
        raise ValueError("Require x>=0, T>0, sigma>0.")
    scale = sigma*np.sqrt(T)
    log1 = log_ndtr((-x-nu*T)/scale)
    log2 = -2*nu*x/sigma**2 + log_ndtr((-x+nu*T)/scale)
    return np.exp(np.minimum(np.logaddexp(log1, log2), 0.0))

def risk_dx(x, T, nu, sigma):
    # Moderate-domain analytic derivative, used only on the stated audit grid.
    scale = sigma*np.sqrt(T)
    a, b = (-x-nu*T)/scale, (-x+nu*T)/scale
    phi = lambda z: np.exp(-z*z/2)/np.sqrt(2*np.pi)
    return (-phi(a)/scale + np.exp(-2*nu*x/sigma**2) *
            (-2*nu*ndtr(b)/sigma**2-phi(b)/scale))

def density(x, t, nu, sigma):
    if x <= 0 or np.any(np.asarray(t) <= 0) or sigma <= 0:
        raise ValueError("Require x,t,sigma>0.")
    return x/(sigma*np.sqrt(2*np.pi*t**3))*np.exp(
        -(x+nu*t)**2/(2*sigma**2*t))

def killed(z, x, t, nu, sigma):
    z = np.asarray(z)
    if np.any(z < 0) or x <= 0 or t <= 0 or sigma <= 0:
        raise ValueError("Require z>=0 and x,t,sigma>0.")
    return (np.exp(-(z-x-nu*t)**2/(2*sigma**2*t))
            /(sigma*np.sqrt(2*np.pi*t))
            *(-np.expm1(-2*x*z/(sigma**2*t))))

def pde_risk(x, T, nu, sigma, dx, steps, xmax=8.0):
    # Independent finite-difference backward PDE:
    # P_T = nu P_x + D P_xx, P(0,T)=1, P(xmax,T)=0, P(x,0)=0.
    # Finite xmax is an approximation; it is far beyond diffusion support here.
    N = round(xmax/dx)
    dx = xmax/N
    n, D = N-1, sigma*sigma/2
    lower, upper = D/dx**2-nu/(2*dx), D/dx**2+nu/(2*dx)
    M = diags([np.full(n-1,lower), np.full(n,-2*D/dx**2),
               np.full(n-1,upper)], [-1,0,1],format="csc")
    I = diags([np.ones(n)],[0],format="csc")
    dt = T/steps
    force = np.zeros(n)
    force[0] = lower
    lhs = splu(I-dt*M/2)
    v = np.zeros(n)
    # Two backward Euler half steps replace the first CN step (Rannacher).
    for _ in range(2):
        v = lhs.solve(v+dt*force/2)
    rhs = I+dt*M/2
    for _ in range(steps-1):
        v = lhs.solve(rhs@v+dt*force)
    return float(np.interp(x,np.linspace(0,xmax,N+1),np.r_[1,v,0]))

print({"python":platform.python_version(), "numpy":np.__version__,
       "scipy":scipy.__version__, "matplotlib":matplotlib.__version__})


In [ ]:
r, delta, sigma_e, cost = 0.05, 0.03, 0.4, 0.08
nu_e = r-delta-sigma_e**2/2
gamma = (nu_e+np.sqrt(nu_e**2+2*r*sigma_e**2))/sigma_e**2
boundary = gamma*cost/(r*(1+gamma))
z = np.geomspace(1.0,20.0,100)
V = boundary*z
C, Q = channels(z,gamma)
E = boundary*C
EV = 1-z**(-gamma-1)
EVV = (gamma+1)/boundary*z**(-gamma-2)
residual = ((r-delta)*V*EV+0.5*sigma_e**2*V**2*EVV
            +delta*V-cost-r*E)

x, T, nu, sig = 0.8, 0.9, -0.3, 0.7
pt = float(risk(x,T,nu,sig))
coarse = pde_risk(x,T,nu,sig,0.02,800)
fine = pde_risk(x,T,nu,sig,0.01,1600)
mass = quad(lambda zz:killed(zz,x,T,nu,sig),0,np.inf,
            epsabs=1e-12,epsrel=1e-12)[0]
fint = quad(lambda tt:density(x,tt,nu,sig),0,T,
            epsabs=1e-12,epsrel=1e-12)[0]
t1, t2 = 0.35, 0.55
composition = float(risk(x,t1,nu,sig))+quad(
    lambda zz:killed(zz,x,t1,nu,sig)*risk(zz,t2,nu,sig),
    0,np.inf,epsabs=1e-12,epsrel=1e-12)[0]
eps = 1e-5
flux_numeric = sig**2/2 * (4*killed(eps,x,T,nu,sig)
                           -killed(2*eps,x,T,nu,sig))/(2*eps)
flux_exact = density(x,T,nu,sig)
gradient_errors = []
for xx in [0.05,0.2,0.8,1.6]:
    approx = (risk(xx+eps,T,nu,sig)-risk(xx-eps,T,nu,sig))/(2*eps)
    gradient_errors.append(abs(float(approx-risk_dx(xx,T,nu,sig))))

results = {
    "scope":"Scalar mathematical audit only; no HCSN training or performance claim",
    "simplified_model":{
        "r":r,"delta":delta,"sigma":sigma_e,"cost":cost,
        "gamma":float(gamma),"boundary":float(boundary),
        "max_ode_residual":float(np.max(np.abs(residual))),
        "C_boundary_derivative":0.0,"Q_right_boundary_derivative":-float(gamma)},
    "risk_model":{"x":x,"T":T,"nu":nu,"sigma":sig},
    "risk_analytic":pt,"risk_pde_coarse":coarse,"risk_pde_fine":fine,
    "pde_abs_errors":[abs(coarse-pt),abs(fine-pt)],
    "mass_error":abs(mass+pt-1),
    "cdf_density_integral_error":abs(fint-pt),
    "semigroup_composition_error":abs(composition-pt),
    "boundary_flux_numeric":float(flux_numeric),
    "boundary_flux_exact":float(flux_exact),
    "boundary_flux_abs_error":float(abs(flux_numeric-flux_exact)),
    "max_risk_dx_finite_difference_error":max(gradient_errors),
    "zero_drift_risk_right_derivative_at_boundary_sigma_0_6_T_1":
        -math.sqrt(2/math.pi)/0.6}
assert np.max(np.abs(residual)) < 1e-12
assert abs(fine-pt) < abs(coarse-pt)
assert abs(fine-pt) < 2e-5
assert abs(mass+pt-1) < 1e-10
assert abs(fint-pt) < 1e-10
assert abs(composition-pt) < 1e-10
assert abs(flux_numeric-flux_exact) < 1e-8
assert max(gradient_errors) < 1e-8
import json
print(json.dumps(results,indent=2,ensure_ascii=False))


## 5. 核验图：静态双通道、导数、风险与时间流

左上、右上用常成本简化；左下、右下用固定系数首达过程。右下是首达时间密度的单次脉冲形状，**不是已验证的循环神经元或生物动作电位**。图中的负安全裕度风险=1延拓用于展示折点，未被当作原公式在负域的解析延伸。


In [ ]:
plt.rcParams.update({"font.size":10,"axes.spines.top":False,
                     "axes.spines.right":False})
fig, axs = plt.subplots(2,2,figsize=(11,7.2),layout="constrained")
zz = np.linspace(0.7,4,450)
safe = np.maximum(zz,1)
cc,qq = channels(safe,gamma)
axs[0,0].plot(zz,cc,label="Continuation feature C",color="#165dba",lw=2.5)
axs[0,0].plot(zz,qq,label="Discounted hit feature Q",color="#bd4931",lw=2.5)
axs[0,0].axvline(1,color="gray",ls="--",lw=1)
axs[0,0].set(xlabel="State / boundary (z)",ylabel="Dimensionless feature",
             title="Two analytic features, one state variable")
axs[0,0].legend()
dc = np.where(zz>1,1-safe**(-gamma-1),0)
dq = np.where(zz>1,-gamma*safe**(-gamma-1),0)
axs[0,1].plot(zz,dc,label="dC/dz",color="#165dba",lw=2.5)
axs[0,1].plot(zz,dq,label="dQ/dz (piecewise)",color="#bd4931",lw=2.5)
axs[0,1].axvline(1,color="gray",ls="--",lw=1)
axs[0,1].set(xlabel="State / boundary (z)",ylabel="Derivative",
             title="C joins smoothly; Q has a derivative jump")
axs[0,1].legend()
xx = np.linspace(-0.25,2,400)
for horizon,color in [(0.25,"#7595b5"),(1,"#165dba"),(4,"#1a8070")]:
    vals = risk(np.maximum(xx,0),horizon,0,0.6)
    axs[1,0].plot(xx,vals,label=f"T={horizon}",color=color,lw=2)
axs[1,0].axvline(0,color="gray",ls="--",lw=1)
axs[1,0].set(xlabel="Log safety margin x",ylabel="First-hit probability",
             title="A risk forecast depends on its horizon")
axs[1,0].legend()
tt = np.linspace(0.01,4,500)
for xx0,color in [(0.4,"#7595b5"),(0.8,"#165dba"),(1.2,"#1a8070")]:
    axs[1,1].plot(tt,density(xx0,tt,-0.3,0.7),label=f"x0={xx0}",
                  color=color,lw=2)
axs[1,1].set(xlabel="Elapsed time",ylabel="Boundary probability flux",
             title="First-hit density is a flow, not a threshold")
axs[1,1].legend()
fig.suptitle("HCSN mathematical audit — restricted scalar models",fontsize=15)
plt.show()


## 6. 真正的 Continuous Free-Boundary Flow 可以怎样定义

### 轻量连续状态版本

保存证据、边界/资源和恢复状态，以 C、Q 或有限期限 P 作可微特征，输出至少包含语义 value vector 与风险/裕度信息。需要另外定义稳定状态更新及慢恢复。它可以是有用的连续时间循环网络，但其解析特征不等于其全网络动力学解析，概率读出也不等于实际二值发放。

一个尚未训练或验证兴奋性的具体候选是

$$x=\varepsilon+\operatorname{softplus}(u),\quad
c(x)=\frac{C(e^x)}{1+C(e^x)},\quad p(x)=P_T(x),$$
$$\tau_u\dot u=-[\kappa_0+\kappa_Cc(x)](u-u_0)-g\,p(x)+a-I(t),$$
$$\tau_a\dot a=-a+\eta p(x).$$

设 $\kappa_0,\varepsilon,T,\sigma>0$，其余增益非负，时间常数正，初值有限且输入有界，则 p、c 有界，a 有界，带正下界的线性恢复系数保证 u 有界。正 I 降低安全裕度，风险再生项 -gp 继续降低裕度，慢变量 a 产生恢复。Call 特征仅调节一个已经具有明确回复方向的项。需要另行检验静息稳定性、快慢零流线、脉冲样响应及分岔；有界性本身不证明兴奋性。

由于实际 x 并不服从上节的常漂移 Brownian motion，这里的 p 是解析风险形状激活，通常不是该新ODE自身的精确首达概率。softplus 也令 x 始终大于零，故应称连续近阈值响应。

如果仅把 E、P 放进一个 MLP，创新应按解析非线性与门控结构判断。应与相同参数/状态预算的 sigmoid、SiLU、单调样条、GRU、CfC 对比；不能仅因采用金融名称就视为新机制。

### 保留发放意义的概率流版本

在固定边界坐标 x>0，保存活动状态密度 p：

$$\partial_t p=-\partial_x(\nu p)+\tfrac{\sigma^2}{2}\partial_{xx}p
+j(t-\tau_{\rm ref})\delta(x-x_r),\quad p(0,t)=0,$$
$$j(t)=\tfrac{\sigma^2}{2}\partial_xp(0+,t).$$

假定复位到固定 $x_r>0$、固定不应期、此期间不接受输入。取初始活动质量1、初始不应期质量0，且 $j(t)=0$ 对所有 $t<0$；或给出一致的既往发放历史。令 $M(t)=\int_0^\infty p(x,t)dx$，$R(t)=\int_{t-\tau_{\rm ref}}^t j(s)ds$，有 $M'=-j+j(t-\tau_{\rm ref})$、$R'=j-j(t-\tau_{\rm ref})$，因此活动密度与不应期质量之和保持1。零不应期时注入项为 $j(t)\delta(x-x_r)$。常参数下，重复发放流可写成首达密度的 renewal 卷积方程。

这种输出是群体/期望发放流，不是单个随机神经元样本路径的无损替代。尤其非线性后续层中，一般 $E[g(S)]\ne g(E[S])$；平均流训练后采样真实事件可能改变任务分布。

若要离散真实事件，需要事件梯度、重参数化或 score-function 等方法；不能仅靠连续 CDF 的自动微分忽略 reset 与路径概率。

### 加入复位会改变自由边界问题

继续价值 J、运行收益 l、折现 rho 和干预算子 M 应满足

$$\max\{\mathcal L J-\rho J+\ell,\;\mathcal M J-J\}=0.$$

例如 $\mathcal M J=R_{\rm emit}-c_{\rm emit}
+e^{-\rho\tau_{\rm ref}}J(x_r)$。边界要与发放后的继续价值拼接；一般不再满足原企业终止问题的 J(b)=J'(b)=0。应重新求解，或声明原边界仅为启发式门控。


## 7. BP 可行的条件

用户所说 BF 若指误差反向传播，这里使用通常名称 BP。

内域闭式函数可微，标准离散训练可用 BPTT；连续方程可用求解器自动微分或伴随法。全网络长期梯度仍受 Jacobian 连乘、时间常数、数值精度、饱和与分岔影响。

对候选边界 b 的 continuation 解 $E(V;b,\theta)$，可定义

$$G(b,\theta)=E_V(V;b,\theta)|_{V=b}=0,\qquad
\frac{db}{d\theta}=-G_b^{-1}G_\theta.$$

需要边界解局部唯一且 $G_b\ne0$。这里 $G_b$ 包含自变量 V=b 和 E 对边界参数 b 的全部依赖，不能随意只保留 $E_{VV}$。仅对 E(b)=0 使用 $-E_\theta/E_V$ 会在 smooth-pasting 点遇到零分母。

工程上先约束正的时间/噪声尺度，采用 log-CDF、expm1、解析极限和混合精度检查。若把 $\sigma\to0$、$T\to0$ 作为训练目标，可能重新逼近硬阶跃及病态梯度。

已有 EventProp 对特定硬阈值 LIF 网络给出几乎处处精确的事件梯度；“SNN 原本不能 BP”不是成立的比较前提。[2]


## 8. 与 SNN、Transformer 及全模态 encoder 的比较

| 方向 | 候选收益 | 成立条件与代价 |
|---|---|---|
| 相对硬事件 SNN | 连续特征可避免部分替代梯度偏差 | 若保留事件/reset仍须正确事件梯度；连续 dense 通信失去二值稀疏成本优势 |
| 连续时间/异步输入 | 时间间隔、风险期限及恢复可显式建模 | CfC、Neural CDE、ALIF 已有同类动机；需要强基线 |
| 流式长序列 | 固定状态推理可能减少随历史长度增长的存储 | BPTT训练仍有轨迹成本；压缩状态不保证任意历史检索 |
| 相对全注意力 | 若无全对全交互，序列长度上的核心递推可线性 | dense投影常有 O(Td²)，CDF/根求解/积分均收费；非线性递推难直接并行scan |
| 全模态 | 适合流式音频/事件视觉/传感器的动态融合 | 语义编码、跨模态对齐、实体/时间一致性与decoder必须另行训练 |
| 多Cell共享encoder | 动态层/Head可低成本做隔离实验 | encoder共享节约属于共享方案自身，不能直接归功于HCSN |

FlashAttention 已优化精确注意力的IO与中间存储，不应把朴素 T×T 矩阵实现当作唯一 Transformer 基线。[4] CfC 已有解析近似连续网络。[3] 近期状态空间基线可包括 Mamba-3，其论文研究检索、状态跟踪与实际推理效率。[5]

推荐首先冻结共享 encoder，以完全相同特征训练不同动态层/Head；只有确认任务收益独立于额外状态量、信息和训练预算后，再做混合 backbone 与规模扩展。


## 9. 历史证据、实验门槛和研究结论

本次读取了已保存的 RR_SNN_V2_Report.ipynb。该相关、但不同于本次连续流 HCSN 的候选记录：真实脉冲反馈五种子实验，在相同 CPU 预算目标的合成 shift 上，候选减对照为 **−10.76 [−13.41, −8.21] 个百分点**。本次未重跑其原始训练；该数字是已有报告记载，不能当成新的独立复现，也不能用它否定所有未来连续流架构。[8]

建议验证顺序：

1. 原文完整复现：债务解、equity A7、边界12、零流动性等极限，对照独立 PDE。
2. 神经动力学闭合：刺激响应、恢复、不应期、重复输出、耦合稳定性及步长收敛。
3. 同预算小任务：异步采样、长程记忆、检索、缺失/噪声输入，多随机种子与样本外评价。
4. 同一冻结 encoder：普通门控、GRU、ALIF、CfC、SSM、attention Head 公平对照。
5. 混合 backbone：语义 value 支路 + 风险/边界门控，保留检索路径；含实际特殊函数成本。
6. 扩模：仅在质量—成本 Pareto 改善持续成立后扩到更大 token 与参数预算。

预先固定评价集、选择指标、同计算预算和失败保留规则。需要实际测量 tokens/s、时延p95/p99、峰值显存/RSS、训练时长和焦耳/任务，不能从解析式或脉冲数直接推算优势。

**结论：数学原语可构造，普通BP路径可构造，具有恢复/复位的完整模型也有可行路线；特定自由边界结构是否增加任务价值、能否扩展为全模态大模型，仍须公平实验。**

## 10. 一手资料与已保存的前序研究

1. He, Z. & Xiong, W. (2012). Rollover Risk and Credit Risk. https://www.princeton.edu/~wxiong/papers/rollover.pdf
2. Wunderlich, T. C. & Pehle, C. (2021). Event-based backpropagation can compute exact gradients for spiking neural networks. https://pmc.ncbi.nlm.nih.gov/articles/PMC8213775/
3. Hasani et al. (2022). Closed-form continuous-time neural networks. https://www.nature.com/articles/s42256-022-00556-7
4. Dao et al. (2022). FlashAttention. https://arxiv.org/abs/2205.14135
5. Lahoti et al. (2026). Mamba-3. https://arxiv.org/abs/2603.15569
6. FitzHugh (1961). Impulses and Physiological States in Theoretical Models of Nerve Membrane. https://pmc.ncbi.nlm.nih.gov/articles/PMC1366333/
7. Gerstein & Mandelbrot (1964). Random Walk Models for the Spike Activity of a Single Neuron. https://pubmed.ncbi.nlm.nih.gov/14104072/
8. RR_SNN_V2_Report.ipynb (2026-10-29), saved prior report, read this turn: [private Library reference omitted from public source]
9. rollover_snn_research.ipynb (2026-10-29): [private Library reference omitted from public source]
